In [3]:

import pandas as pd
import numpy as np
import os
import json
import pickle
import time
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Vector store
import faiss

# Embeddings (same as Task 2)
from sentence_transformers import SentenceTransformer

# For LLM
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import torch

# For evaluation
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

print("Loded successfully")

Loded successfully


In [4]:

print("\n Loading embedding model...")
try:
    # Try to load sentence transformer
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    print(" Loaded all-MiniLM-L6-v2")
except:
    # Fallback to local model if available
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.decomposition import TruncatedSVD
        from sklearn.preprocessing import normalize
        
        class LocalEmbeddingModel:
            def __init__(self, dimension=384):
                self.dimension = dimension
                self.vectorizer = None
                self.svd = None
                self.is_fitted = False
                
            def encode(self, texts, normalize_embeddings=True):
                if isinstance(texts, str):
                    texts = [texts]
                
                if not self.is_fitted:
                    # Fit on the fly
                    self.vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
                    self.vectorizer.fit(texts)
                    tfidf = self.vectorizer.transform(texts)
                    n_components = min(self.dimension, tfidf.shape[1] - 1)
                    self.svd = TruncatedSVD(n_components=n_components, random_state=42)
                    self.svd.fit(tfidf)
                    self.is_fitted = True
                
                tfidf = self.vectorizer.transform(texts)
                reduced = self.svd.transform(tfidf)
                
                if reduced.shape[1] < self.dimension:
                    padded = np.zeros((reduced.shape[0], self.dimension))
                    padded[:, :reduced.shape[1]] = reduced
                    reduced = padded
                
                if normalize_embeddings:
                    reduced = normalize(reduced, norm='l2')
                
                return reduced.astype(np.float32)
            
            def get_sentence_embedding_dimension(self):
                return self.dimension
        
        embedding_model = LocalEmbeddingModel()
        print(" Loaded local TF-IDF + SVD model")
    except:
        raise Exception("No embedding model available")

print(f"   Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

# Load FAISS vector store
print("\n Loading FAISS vector store...")
vector_store_dir = '../vector_store/faiss/'

if not os.path.exists(vector_store_dir):
    vector_store_dir = 'vector_store/faiss/'

# Load FAISS index
faiss_index = faiss.read_index(f"{vector_store_dir}/index.faiss")
print(f" FAISS index loaded with {faiss_index.ntotal:,} vectors")

# Load metadata
with open(f"{vector_store_dir}/metadata.pkl", 'rb') as f:
    metadata = pickle.load(f)
print(f" Metadata loaded with {len(metadata['chunk_text'])} entries")

print(f"\n Vector Store Info:")
print(f"  Total vectors: {faiss_index.ntotal:,}")
print(f"  Embedding dimension: {faiss_index.d}")
print(f"  Metadata fields: {list(metadata.keys())}")


 Loading embedding model...


model_O1.onnx: 100%|██████████| 90.4M/90.4M [02:00<00:00, 749kB/s]
model_O2.onnx: 100%|██████████| 90.3M/90.3M [01:32<00:00, 981kB/s] 
model_O3.onnx: 100%|██████████| 90.3M/90.3M [01:25<00:00, 1.05MB/s]
model_O4.onnx: 100%|██████████| 45.2M/45.2M [00:43<00:00, 1.04MB/s]
model_qint8_arm64.onnx: 100%|██████████| 23.0M/23.0M [00:23<00:00, 995kB/s] 
model_qint8_avx512.onnx: 100%|██████████| 23.0M/23.0M [00:24<00:00, 948kB/s]
model_qint8_avx512_vnni.onnx: 100%|██████████| 23.0M/23.0M [00:22<00:00, 1.02MB/s]
model_quint8_avx2.onnx: 100%|██████████| 23.0M/23.0M [00:23<00:00, 1.00MB/s]
openvino_model.bin: 100%|██████████| 90.3M/90.3M [01:21<00:00, 1.11MB/s]
openvino_model.xml: 211kB [00:00, 183MB/s]
openvino_model_qint8_quantized.bin: 100%|██████████| 22.9M/22.9M [00:20<00:00, 1.11MB/s]
openvino_model_qint8_quantized.xml: 368kB [00:00, 23.1MB/s]
pytorch_model.bin: 100%|██████████| 90.9M/90.9M [01:27<00:00, 1.04MB/s]
sentence_bert_config.json: 100%|██████████| 53.0/53.0 [00:00<?, ?B/s]
special_

 Loaded all-MiniLM-L6-v2
   Embedding dimension: 384

 Loading FAISS vector store...
 FAISS index loaded with 36,399 vectors
 Metadata loaded with 36399 entries

 Vector Store Info:
  Total vectors: 36,399
  Embedding dimension: 384
  Metadata fields: ['chunk_text', 'complaint_id', 'product', 'issue', 'company', 'chunk_index', 'total_chunks', 'date_received']


In [6]:

# OPTION 1: Use a small local model (recommended for local testing)
print("\n Loading local LLM (small model for testing)...")

try:
    # Use a small model that can run locally
    # T5 models use AutoModelForSeq2SeqLM, not AutoModelForCausalLM
    model_name = "google/flan-t5-small"  # Small, fast, works locally
    
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    
    # Create pipeline for text generation
    generator = pipeline(
        "text2text-generation",  # Use text2text-generation for T5
        model=model,
        tokenizer=tokenizer,
        device=-1,  # CPU
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True
    )
    
    print(f" Loaded {model_name}")
    print(f"   Max length: {tokenizer.model_max_length}")
    print(f"   Model type: T5 (Seq2Seq)")
    
    def generate_answer(prompt):
        """Generate answer using local model"""
        try:
            # T5 models need the prompt as input
            response = generator(prompt, max_length=200)[0]['generated_text']
            return response
        except Exception as e:
            return f"Error generating response: {e}"
    
    llm_loaded = True
    
except Exception as e:
    print(f" Failed to load local model: {e}")
    print("\nUsing fallback response generator...")
    llm_loaded = False
    
    # OPTION 2: Simple response generator (no LLM required)
    def generate_answer(prompt):
        """Simple fallback response generator"""
        # Extract question and context from prompt
        parts = prompt.split("Context:")
        if len(parts) > 1:
            context_part = parts[1]
            context = context_part.split("Question:")[0].strip() if "Question:" in context_part else ""
            question = context_part.split("Question:")[-1].strip() if "Question:" in context_part else ""
        else:
            return "I don't have enough information to answer this question."
        
        # Simple keyword-based response
        words = question.lower().split()
        relevant_chunks = []
        for chunk in context.split("."):
            if chunk.strip() and any(word in chunk.lower() for word in words[:3]):
                relevant_chunks.append(chunk.strip())
        
        if relevant_chunks:
            # Take first 3 relevant chunks
            response_chunks = relevant_chunks[:3]
            return f"Based on the complaint data, here's what I found: {' '.join(response_chunks)}"
        else:
            return "I don't have enough information in the retrieved complaints to answer this question."

print(" LLM ready!")

# Test the LLM
print("\n Testing LLM...")
test_prompt = "Question: What are credit card complaints? Answer:"
test_response = generate_answer(test_prompt)
print(f"Test response: {test_response[:100]}...")


LOADING LLM

🔄 Loading local LLM (small model for testing)...


model.safetensors:  95%|█████████▌| 294M/308M [35:15<00:22, 636kB/s]  Error while downloading from https://cas-bridge.xethub.hf.co/xet-bridge-us/63526d7c7e4cc3135fd0f17c/a4b111b77195028aec51e6cf1212f562b7a36941d546d8145b2e501c97d90880?Expires=1782125983&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzYzNTI2ZDdjN2U0Y2MzMTM1ZmQwZjE3Yy9hNGIxMTFiNzcxOTUwMjhhZWM1MWU2Y2YxMjEyZjU2MmI3YTM2OTQxZDU0NmQ4MTQ1YjJlNTAxYzk3ZDkwODgwKiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4MjEyNTk4M319fV19&Signature=MEYCIQDVh-wRkauxWFyNRm39r8JxLIgzsfxm4n1IogUBXTwEFwIhAMECcacEB%7EZ4Vl5shsa-4q%7Eu6%7ElTJups6z10bTZplra9&Key-Pair-Id=K1LYXO563TGWFU&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&X-Xet-Cas-Uid=public&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=cas%2F20260622%2Fus-east-1%2Fs3%2Faws4_request&X-Amz-Date=20260622T095943Z&X-Amz-Expires=3600&X-Amz-Signed

⚠️ Failed to load local model: (MaxRetryError('HTTPSConnectionPool(host=\'cas-bridge.xethub.hf.co\', port=443): Max retries exceeded with url: /xet-bridge-us/63526d7c7e4cc3135fd0f17c/a4b111b77195028aec51e6cf1212f562b7a36941d546d8145b2e501c97d90880?Expires=1782125983&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzYzNTI2ZDdjN2U0Y2MzMTM1ZmQwZjE3Yy9hNGIxMTFiNzcxOTUwMjhhZWM1MWU2Y2YxMjEyZjU2MmI3YTM2OTQxZDU0NmQ4MTQ1YjJlNTAxYzk3ZDkwODgwKiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4MjEyNTk4M319fV19&Signature=MEYCIQDVh-wRkauxWFyNRm39r8JxLIgzsfxm4n1IogUBXTwEFwIhAMECcacEB~Z4Vl5shsa-4q~u6~lTJups6z10bTZplra9&Key-Pair-Id=K1LYXO563TGWFU&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&X-Xet-Cas-Uid=public&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=cas%2F20260622%2Fus-east-1%2Fs3%2Faws4_request&X-Amz-Date=20260622T095943Z&X-Amz-Expires=3600&X-Amz

In [8]:


def retrieve_chunks(query, index, metadata, model, k=5):
    """
    Retrieve top-k relevant chunks for a given query
    
    Args:
        query: User question string
        index: FAISS index
        metadata: Metadata dictionary
        model: Embedding model
        k: Number of results to retrieve (default: 5)
    
    Returns:
        List of retrieved chunks with metadata and scores
    """
    # Generate query embedding
    query_embedding = model.encode([query], normalize_embeddings=True)
    
    # Search the index
    distances, indices = index.search(query_embedding.astype('float32'), k)
    
    # Get results
    results = []
    for i, idx in enumerate(indices[0]):
        if idx < len(metadata['chunk_text']):
            results.append({
                'chunk_text': metadata['chunk_text'][idx],
                'complaint_id': metadata['complaint_id'][idx],
                'product': metadata['product'][idx],
                'issue': metadata['issue'][idx],
                'company': metadata['company'][idx],
                'distance': distances[0][i],
                'similarity': 1 / (1 + distances[0][i])  # Convert to similarity
            })
    
    return results

# Test retriever
print("\n Testing retriever...")
test_query = "What are the most common complaints about credit cards?"
test_results = retrieve_chunks(test_query, faiss_index, metadata, embedding_model, k=5)

print(f" Retrieved {len(test_results)} chunks")
print(f"\nTop result:")
print(f"  Product: {test_results[0]['product']}")
print(f"  Issue: {test_results[0]['issue']}")
print(f"  Similarity: {test_results[0]['similarity']:.4f}")
print(f"  Preview: {test_results[0]['chunk_text'][:150]}...")


 Testing retriever...
 Retrieved 5 chunks

Top result:
  Product: Credit card
  Issue: Closing your account
  Similarity: 0.5000
  Preview: . and i am only a volunteer....
